# BusBooking - Kaggle CPU Load Testing (Pha 2)

**Mục tiêu:** đo cùng một bộ kịch bản tải trên cùng loại máy Kaggle CPU, **trước** và **sau** mỗi cải thiện kiến trúc của Pha 2 (xem `docs/07-phase-2-architecture-improvement.md`, mục 7.2). Hai lần đo chỉ được khác nhau ở **file JAR**.

Notebook chạy Spring Boot (Java 21), PostgreSQL, Mailpit và Locust trong cùng một notebook, không dùng Docker, giống notebook Pha 1.

**Các kịch bản:**

| Tên | Nội dung | Mục liên quan |
|---|---|---|
| `read_unfiltered` | `GET /api/trips` không bộ lọc trên 100.000 chuyến (script Pha 1) | A1 |
| `read_search` | Tìm chuyến theo điểm đi, điểm đến, ngày | A1, A2 |
| `hot_trip` | Dồn toàn bộ lượt đặt vé vào 1 chuyến, kèm một ít tải đọc | A3 |
| `register_smtp_ok` | Đăng ký khi SMTP hoạt động | B1 |
| `register_smtp_hang` | Đăng ký khi SMTP treo | B1 |

**Trước khi chạy:**
1. Build JAR từ đúng commit cần đo: `./mvnw -Dmaven.test.skip=true clean package`. Ghi lại commit đó.
2. Tạo **Private Kaggle Dataset** chứa file JAR (không chọn `.jar.original`) và gắn vào notebook bằng **Add Input**.
3. Script tải nằm trong thư mục `load-tests/` của repository. Notebook tự `git clone` nếu repository công khai. Nếu repository riêng tư, tải thư mục `load-tests/` lên một Private Dataset và gắn vào notebook.
4. Chọn **Accelerator = None (CPU)**, **Internet = On**.
5. Sửa `STAGE` ở cell cấu hình: `before` cho lần đo trước cải thiện, tên khác (ví dụ `after_a1`) cho các lần sau.

**Không upload `.env`, mật khẩu, token hay dữ liệu người dùng thật.** Mọi dữ liệu trong notebook là dữ liệu giả lập sinh bằng script.

## 0. Cấu hình

Giữ nguyên mọi giá trị giữa lần đo trước và sau, chỉ đổi `STAGE` và file JAR.

In [ ]:
import os, sys, platform, subprocess, shutil, time, json, hashlib, secrets, base64, socket, re, statistics
from pathlib import Path

STAGE = os.environ.get('P2_STAGE', 'before')          # 'before', 'after_a1', 'after_all', ...

# Repository chứa script tải (dùng khi không gắn load-tests/ làm Dataset)
REPO_URL = 'https://github.com/BinhPhuc/Bus-booking-system.git'
REPO_REF = 'main'

RUN_SECONDS = int(os.environ.get('P2_RUN_SECONDS', 60))   # thời gian mỗi lượt đo
RUNS = int(os.environ.get('P2_RUNS', 3))                  # số lượt mỗi mức tải; kết quả lấy trung vị
WARMUP_SECONDS = int(os.environ.get('P2_WARMUP_SECONDS', 15))
TRIPS = int(os.environ.get('P2_TRIPS', 100000))           # số chuyến xe sinh ra

# Số người dùng ảo cho từng kịch bản
LEVELS = {
    'read_unfiltered': [1, 5],     # mỗi response chứa toàn bộ chuyến xe nên chỉ đo ở tải thấp
    'read_search': [20, 100],
    'hot_trip': [20, 100, 400],     # 400 để tạo tranh chấp thật trên một chuyến
    'register': [20],
}
HOT_TRIP_COUNT = 1                 # số chuyến "hot" nhận tải đặt vé

# Giới hạn đăng ký mặc định là 5 request/phút/IP; phải nâng lên để đo được kịch bản đăng ký.
AUTH_RATE_REGISTER_LIMIT = 1000000
JVM_OPTIONS = ['-Xms256m', '-Xmx1024m', '-Duser.timezone=UTC']   # giống Pha 1

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT = WORK / f'busbooking_phase2_{STAGE}'
RAW_DIR = OUTPUT / 'raw'            # log và CSV chi tiết của từng lượt; chỉ cần khi phải tra cứu
RAW_DIR.mkdir(parents=True, exist_ok=True)
print('Stage:', STAGE)
print('Output:', OUTPUT)
print('OS:', platform.platform())
print('Python:', sys.version.split()[0])
print('CPU logical:', os.cpu_count())
subprocess.run(['bash', '-lc', r'lscpu | grep -E "Model name|CPU\(s\)|Architecture" | head -6'], check=False)

## 1. Cài công cụ tạo tải và đo tài nguyên

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'locust>=2.30,<3', 'psutil>=6,<9', 'requests>=2.31,<3', 'pandas>=2,<3'], check=True)
import requests, psutil, pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
print('Locust:', subprocess.check_output([sys.executable, '-m', 'locust', '--version'], text=True).strip())
print('psutil:', psutil.__version__)

## 2. Chuẩn bị Java 21

In [ ]:
JAVA_DIR = Path('/tmp/busbooking_jdk21')
JAVA_BIN = JAVA_DIR / 'bin/java'
if not JAVA_BIN.exists():
    archive = Path('/tmp/busbooking_java21.tar.gz')
    url = 'https://corretto.aws/downloads/latest/amazon-corretto-21-x64-linux-jdk.tar.gz'
    subprocess.run(['curl', '-fL', '--retry', '3', '-o', str(archive), url], check=True)
    extract_dir = Path('/tmp/busbooking_jdk21_extract')
    extract_dir.mkdir(exist_ok=True)
    subprocess.run(['tar', '-xzf', str(archive), '-C', str(extract_dir)], check=True)
    candidate = next(p for p in extract_dir.iterdir() if (p / 'bin/java').exists())
    shutil.move(str(candidate), str(JAVA_DIR))
    archive.unlink(missing_ok=True)
assert JAVA_BIN.exists(), 'Chưa tìm được Java 21'
print(subprocess.run([str(JAVA_BIN), '-version'], capture_output=True, text=True, check=True).stderr)

## 3. Khởi động PostgreSQL riêng trong Kaggle

Database `busbooking_kaggle` chỉ tồn tại trong notebook; mật khẩu ngẫu nhiên giữ trong RAM.

In [ ]:
if shutil.which('pg_lsclusters') is None:
    print('Installing PostgreSQL (có thể tốn một ít thời gian)...')
    subprocess.run(['bash', '-lc', 'apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql postgresql-client'], check=True)
clusters = subprocess.check_output(['pg_lsclusters', '--no-header'], text=True).strip().splitlines()
if not clusters:
    raise RuntimeError('PostgreSQL không có cluster; kiểm tra đầu ra apt hoặc tạo cluster bằng pg_createcluster.')
print('Clusters:', clusters)
version, cluster_name, port, status = clusters[0].split()[:4]
if status != 'online':
    subprocess.run(['pg_ctlcluster', version, cluster_name, 'start'], check=True)
subprocess.run(['pg_isready', '-h', '127.0.0.1', '-p', port], check=True)
DB_NAME = 'busbooking_kaggle'
DB_USER = 'busbooking_kaggle_user'
DB_PASSWORD = secrets.token_urlsafe(30)

def admin_sql(sql):
    return subprocess.run(['runuser', '-u', 'postgres', '--', 'psql', '-v', 'ON_ERROR_STOP=1', '-q'], input=sql, text=True, capture_output=True, check=True)

def admin_value(sql):
    return subprocess.check_output(['runuser', '-u', 'postgres', '--', 'psql', '-tAc', sql], text=True).strip()

if admin_value(f"SELECT 1 FROM pg_roles WHERE rolname='{DB_USER}'") == '1':
    admin_sql(f"ALTER ROLE {DB_USER} WITH LOGIN PASSWORD '{DB_PASSWORD}';")
else:
    admin_sql(f"CREATE ROLE {DB_USER} WITH LOGIN PASSWORD '{DB_PASSWORD}';")
if admin_value(f"SELECT 1 FROM pg_database WHERE datname='{DB_NAME}'") != '1':
    subprocess.run(['runuser', '-u', 'postgres', '--', 'createdb', '-O', DB_USER, DB_NAME], check=True)

PG_ENV = os.environ.copy()
PG_ENV['PGPASSWORD'] = DB_PASSWORD
PSQL = ['psql', '-h', '127.0.0.1', '-p', port, '-U', DB_USER, '-d', DB_NAME]

def db_value(sql):
    return subprocess.run(PSQL + ['-X', '-tA', '-v', 'ON_ERROR_STOP=1'], input=sql, text=True, capture_output=True, env=PG_ENV, check=True).stdout.strip()

POSTGRES_VERSION = subprocess.check_output(['psql', '--version'], text=True).strip()
print('Database ready:', DB_NAME, 'at localhost:' + port, '(credentials omitted)')
print('PostgreSQL:', POSTGRES_VERSION)

## 4. Chọn Spring Boot JAR

Đây là thứ duy nhất thay đổi giữa lần đo trước và sau. SHA256 được ghi vào manifest.

In [ ]:
JAR_OVERRIDE = os.environ.get('P2_JAR', '')  # Ví dụ: '/kaggle/input/busbooking-jar/busbooking-0.0.1-SNAPSHOT.jar'
INPUT = Path('/kaggle/input')
jar_candidates = sorted(p for p in INPUT.rglob('*.jar') if not p.name.endswith(('.jar.original', '-sources.jar', '-javadoc.jar'))) if INPUT.exists() else []
print('JAR candidates:', *map(str, jar_candidates), sep='\n - ')
if JAR_OVERRIDE:
    JAR_PATH = Path(JAR_OVERRIDE)
elif len(jar_candidates) == 1:
    JAR_PATH = jar_candidates[0]
else:
    raise RuntimeError('Gắn Kaggle Dataset chứa duy nhất một Spring Boot JAR hoặc chỉnh JAR_OVERRIDE.')
assert JAR_PATH.is_file()
import zipfile
with zipfile.ZipFile(JAR_PATH) as z:
    assert any(n.startswith('BOOT-INF/') for n in z.namelist()), 'JAR này không phải Spring Boot executable JAR; hãy dùng Maven package.'
JAR_SHA256 = hashlib.sha256(JAR_PATH.read_bytes()).hexdigest()
print('Selected JAR:', JAR_PATH.name)
print('SHA256:', JAR_SHA256)

## 5. Lấy script tải

Ưu tiên thư mục `load-tests/` đã gắn bằng **Add Input**; nếu không có thì `git clone` repository. Script được chép vào thư mục làm việc vì `/kaggle/input` chỉ đọc.

In [ ]:
NEEDED = {
    'datasets': ['large_dataset.sql', 'large_dataset_cleanup.sql', 'seed_large_dataset.sh'],
    'scripts': ['make_tokens.py', 'get_trip_locust.py', 'search_trips_locust.py',
                'hot_trip_booking_locust.py', 'register_locust.py', 'smtp_blackhole.py'],
}
LT = WORK / 'load-tests'
SCRIPTS_SOURCE = None
SCRIPTS_COMMIT = None

def find_in(root, name):
    return next((p for p in sorted(root.rglob(name)) if p.is_file()), None)

source_root = Path(os.environ['P2_LOADTEST_DIR']) if os.environ.get('P2_LOADTEST_DIR') else INPUT
found = {name: find_in(source_root, name) for names in NEEDED.values() for name in names} if source_root.exists() else {}
if found and all(found.values()):
    SCRIPTS_SOURCE = f'input: {source_root}'
else:
    clone_dir = Path('/tmp/busbooking_repo')
    shutil.rmtree(clone_dir, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(clone_dir)], check=True)
    SCRIPTS_COMMIT = subprocess.check_output(['git', '-C', str(clone_dir), 'rev-parse', 'HEAD'], text=True).strip()
    found = {name: find_in(clone_dir / 'load-tests', name) for names in NEEDED.values() for name in names}
    SCRIPTS_SOURCE = f'git: {REPO_URL}@{REPO_REF}'
missing = [name for name, path in found.items() if path is None]
if missing:
    raise RuntimeError(f'Thiếu script tải: {missing}. Hãy push thư mục load-tests/ lên {REPO_REF} hoặc gắn nó bằng Add Input.')

SCRIPT_HASHES = {}
for folder, names in NEEDED.items():
    (LT / folder).mkdir(parents=True, exist_ok=True)
    for name in names:
        shutil.copyfile(found[name], LT / folder / name)
        SCRIPT_HASHES[name] = hashlib.sha256((LT / folder / name).read_bytes()).hexdigest()[:16]
print('Scripts from:', SCRIPTS_SOURCE, '| commit:', SCRIPTS_COMMIT)
for name, digest in SCRIPT_HASHES.items():
    print(f'  {digest}  {name}')

## 6. Khởi động BusBooking

Backend chạy ở `127.0.0.1:8085` với database Kaggle. `JWT_SECRET` sinh ngẫu nhiên và chỉ nằm trong RAM. Hàm `ensure_backend()` khởi động lại backend nếu nó chết giữa các lượt đo.

In [ ]:
API_BASE = 'http://127.0.0.1:8085'
JWT_SECRET = base64.b64encode(secrets.token_bytes(48)).decode()
app_env = os.environ.copy()
app_env.update({
    'SERVER_PORT': '8085',
    'DB_URL': f'jdbc:postgresql://127.0.0.1:{port}/{DB_NAME}',
    'DB_USERNAME': DB_USER,
    'DB_PASSWORD': DB_PASSWORD,
    'JWT_SECRET': JWT_SECRET,
    'JWT_ISSUER': 'busbooking',
    'EMAIL_OTP_SECRET': base64.b64encode(secrets.token_bytes(48)).decode(),
    'BOOTSTRAP_SUPER_ADMIN_ENABLED': 'false',
    'SMTP_HOST': '127.0.0.1',
    'SMTP_PORT': '1025',
    'SMTP_AUTH': 'false',
    'SMTP_STARTTLS': 'false',
    'AUTH_RATE_REGISTER_LIMIT': str(AUTH_RATE_REGISTER_LIMIT),
})
backend_log = RAW_DIR / 'backend.log'
backend_proc = None
backend_starts = 0

def backend_alive():
    if backend_proc is None or backend_proc.poll() is not None:
        return False
    try:
        return requests.get(API_BASE + '/api/locations', timeout=5).status_code == 200
    except requests.RequestException:
        return False

def stop_backend():
    global backend_proc
    if backend_proc is not None and backend_proc.poll() is None:
        backend_proc.terminate()
        try:
            backend_proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            backend_proc.kill()
            backend_proc.wait()
    backend_proc = None

def start_backend():
    global backend_proc, backend_starts
    stop_backend()
    handle = open(backend_log, 'a', encoding='utf-8')
    backend_proc = subprocess.Popen([str(JAVA_BIN), *JVM_OPTIONS, '-jar', str(JAR_PATH)],
                                    env=app_env, cwd=str(WORK), stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
    handle.close()
    backend_starts += 1
    deadline = time.monotonic() + 150
    while time.monotonic() < deadline and backend_proc.poll() is None:
        if backend_alive():
            print('Backend started, PID =', backend_proc.pid)
            return
        time.sleep(2)
    print('\n'.join(backend_log.read_text(errors='replace').splitlines()[-60:]))
    raise RuntimeError('Backend chưa start đúng. Kiểm tra env/Flyway trong log ở trên.')

def ensure_backend():
    """Trả về True nếu phải khởi động lại backend."""
    if backend_alive():
        return False
    print('Backend không phản hồi, khởi động lại...')
    start_backend()
    return True

start_backend()
actuator = requests.get(API_BASE + '/actuator/prometheus', timeout=10)
HAS_METRICS = actuator.status_code == 200
print('Actuator metrics:', 'có' if HAS_METRICS else f'không ({actuator.status_code}); bỏ qua số đo pool và khóa')

## 7. Sinh dữ liệu lớn và ký JWT

Script sinh chuyến xe, booking, tài khoản `USER` đã xác minh và các chuyến "hot". JWT được ký trực tiếp bằng `JWT_SECRET` của notebook, không gọi API đăng nhập (rate limiter chỉ cho 10 lượt/phút/IP).

**Chỉ chạy trên database Kaggle của notebook.**

In [ ]:
seed_env = PG_ENV.copy()
seed_env['PSQL'] = ' '.join(PSQL)
seed_env['TRIPS'] = str(TRIPS)
already_seeded = db_value("SELECT count(*) FROM users WHERE email LIKE 'lt-user-%@loadtest.example'") != '0'
t0 = time.monotonic()
subprocess.run(['bash', str(LT / 'datasets/seed_large_dataset.sh')] + (['--reset'] if already_seeded else []), env=seed_env, check=True)
print(f'Seed time: {time.monotonic() - t0:.1f}s')

MANIFEST_PATH = LT / 'datasets/generated/large_dataset_manifest.json'
TOKENS_PATH = LT / 'datasets/generated/tokens.json'
token_env = os.environ.copy()
token_env['JWT_SECRET'] = JWT_SECRET
subprocess.run([sys.executable, str(LT / 'scripts/make_tokens.py')], env=token_env, check=True)

DATA = json.loads(MANIFEST_PATH.read_text())
DATASET_COUNTS = {
    'trips': int(db_value('SELECT count(*) FROM trips')),
    'bookings': int(db_value('SELECT count(*) FROM bookings')),
    'users': int(db_value('SELECT count(*) FROM users')),
    'routes': int(db_value('SELECT count(*) FROM routes')),
}
print('Dataset:', DATASET_COUNTS)
print('Hot trips:', DATA['hot_trip_ids'], '| sample trip:', DATA['sample_trip_id'])

first_token = json.loads(TOKENS_PATH.read_text())[0]
me = requests.get(API_BASE + '/api/users/me', headers={'Authorization': f'Bearer {first_token}'}, timeout=10)
assert me.status_code == 200, f'JWT tự ký không được chấp nhận: {me.status_code} {me.text[:200]}'
detail = requests.get(f"{API_BASE}/api/trips/{DATA['sample_trip_id']}", timeout=10)
assert detail.status_code == 200, f'GET trip detail: {detail.status_code}'
print('JWT: OK | trip detail: OK')

## 8. Hàm chạy một kịch bản

Mỗi mức tải gồm một lượt khởi động không ghi kết quả, rồi `RUNS` lượt đo. Bảng tổng hợp lấy **trung vị** của các lượt.

Trong lúc chạy, notebook lấy mẫu:
- CPU và RAM của tiến trình Java (như Pha 1);
- từ `/actuator/prometheus` (nếu JAR có Actuator): số kết nối HikariCP đang dùng và đang chờ, thời gian chờ khóa dòng chuyến xe.

Locust, PostgreSQL và backend dùng chung CPU của notebook, nên chỉ so sánh với lần đo dùng đúng notebook này.

In [ ]:
METRIC_LINE = re.compile(r'^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{[^}]*\})?\s+([0-9eE+\-.]+|NaN)\s*$')

def scrape_metrics():
    """Trả về {tên metric: [(nhãn, giá trị)]} hoặc {} nếu không lấy được."""
    if not HAS_METRICS:
        return {}
    try:
        text = requests.get(API_BASE + '/actuator/prometheus', timeout=3).text
    except requests.RequestException:
        return {}
    wanted = ('hikaricp_connections_active', 'hikaricp_connections_pending',
              'booking_trip_lock_wait_seconds_sum', 'booking_trip_lock_wait_seconds_count')
    out = {}
    for line in text.splitlines():
        if not line.startswith(wanted):
            continue
        match = METRIC_LINE.match(line)
        if match and match.group(1) in wanted:
            out.setdefault(match.group(1), []).append((match.group(2) or '', float(match.group(3))))
    return out

def metric_total(metrics, name, label_filter=''):
    return sum(value for labels, value in metrics.get(name, []) if label_filter in labels)

def locust_run(tag, locust_file, users, seconds, env_extra):
    """Chạy một lượt Locust, trả về (dict số đo chung, DataFrame thống kê theo request)."""
    prefix = RAW_DIR / tag
    env = os.environ.copy()
    env.update(env_extra)
    spawn_rate = min(users, max(5, users // 10))
    cmd = [sys.executable, '-m', 'locust', '-f', str(locust_file), '--headless', '--host', API_BASE,
           '-u', str(users), '-r', str(spawn_rate), '-t', f'{seconds}s',
           '--csv', str(prefix), '--only-summary', '--stop-timeout', '5']
    java_proc = psutil.Process(backend_proc.pid)
    before = scrape_metrics()
    samples, pool_active, pool_pending = [], [], []
    with open(RAW_DIR / (tag + '.console.log'), 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
        try:
            java_proc.cpu_percent(interval=None)
        except psutil.Error:
            pass
        tick = 0
        while p.poll() is None:
            try:
                samples.append((java_proc.cpu_percent(interval=None), java_proc.memory_info().rss / 1024**2))
            except psutil.Error:
                pass
            if tick % 2 == 0:
                current = scrape_metrics()
                if current:
                    pool_active.append(metric_total(current, 'hikaricp_connections_active'))
                    pool_pending.append(metric_total(current, 'hikaricp_connections_pending'))
            tick += 1
            time.sleep(1)
        rc = p.wait()
    after = scrape_metrics()
    stats_file = RAW_DIR / (tag + '_stats.csv')
    if not stats_file.exists():
        raise RuntimeError(f'Locust failed for {tag}: {(RAW_DIR / (tag + ".console.log")).read_text()[-2500:]}')
    stats = pd.read_csv(stats_file)
    lock_count = metric_total(after, 'booking_trip_lock_wait_seconds_count', 'operation="create"') - metric_total(before, 'booking_trip_lock_wait_seconds_count', 'operation="create"')
    lock_sum = metric_total(after, 'booking_trip_lock_wait_seconds_sum', 'operation="create"') - metric_total(before, 'booking_trip_lock_wait_seconds_sum', 'operation="create"')
    nan = float('nan')
    common = dict(
        java_cpu_avg_pct=sum(s[0] for s in samples) / len(samples) if samples else nan,
        java_rss_max_mb=max(s[1] for s in samples) if samples else nan,
        pool_active_max=max(pool_active) if pool_active else nan,
        pool_pending_max=max(pool_pending) if pool_pending else nan,
        lock_wait_avg_ms=(1000 * lock_sum / lock_count) if lock_count > 0 else nan,
        locust_exit_code=rc,
    )
    return common, stats

RAW = []   # một dòng cho mỗi (kịch bản, mức tải, lượt, request)

def run_scenario(scenario, locust_file, users_levels, env_extra):
    for users in users_levels:
        if WARMUP_SECONDS > 0:
            ensure_backend()
            locust_run(f'{scenario}_u{users}_warmup', locust_file, users, WARMUP_SECONDS, env_extra)
        for run in range(1, RUNS + 1):
            restarted_before = ensure_backend()
            tag = f'{scenario}_u{users}_run{run}'
            print(f'===== {tag}: {users} users / {RUN_SECONDS}s =====')
            common, stats = locust_run(tag, locust_file, users, RUN_SECONDS, env_extra)
            died = not backend_alive()
            for _, row in stats.iterrows():
                name = str(row['Name'])
                count = int(row['Request Count'])
                failures = int(row['Failure Count'])
                RAW.append(dict(
                    scenario=scenario, users=users, run=run,
                    request='ALL' if name.lower() == 'aggregated' else name,
                    requests=count, failures=failures,
                    error_pct=100 * failures / count if count else float('nan'),
                    rps=float(row['Requests/s']), avg_ms=float(row['Average Response Time']),
                    p50_ms=float(row['50%']), p95_ms=float(row['95%']), p99_ms=float(row['99%']),
                    backend_restarted_before=restarted_before, backend_died=died, **common))
            agg = RAW[-1]
            print(f"  rps={agg['rps']:.1f} p95={agg['p95_ms']:.0f}ms p99={agg['p99_ms']:.0f}ms errors={agg['error_pct']:.2f}% "
                  f"cpu={agg['java_cpu_avg_pct']:.0f}% pool_pending_max={agg['pool_pending_max']} died={died}")
    pd.DataFrame(RAW).to_csv(RAW_DIR / 'phase2_runs.csv', index=False)

## 9. Kịch bản `read_unfiltered`: `GET /api/trips` không bộ lọc

Dùng nguyên script Pha 1 (`get_trip_locust.py`) trên dữ liệu lớn. Trước A1, mỗi response chứa toàn bộ chuyến xe, nên kết quả rất chậm hoặc backend hết bộ nhớ là điều được dự đoán; notebook ghi lại và tự khởi động lại backend.

In [ ]:
run_scenario('read_unfiltered', LT / 'scripts/get_trip_locust.py', LEVELS['read_unfiltered'],
             {'BUSBOOKING_TRIP_ID': str(DATA['sample_trip_id'])})

## 10. Kịch bản `read_search`: tìm chuyến theo điểm đi, điểm đến, ngày

In [ ]:
run_scenario('read_search', LT / 'scripts/search_trips_locust.py', LEVELS['read_search'],
             {'BUSBOOKING_MANIFEST': str(MANIFEST_PATH)})

## 11. Kịch bản `hot_trip`: dồn lượt đặt vé vào chuyến "hot"

80% người dùng ảo đặt vé trên chuyến "hot", mỗi người một tài khoản; 20% đọc chi tiết chuyến để xem tranh chấp khóa có làm chậm request khác không. Sau khi chạy, kiểm tra cân bằng ghế.

In [ ]:
run_scenario('hot_trip', LT / 'scripts/hot_trip_booking_locust.py', LEVELS['hot_trip'],
             {'BUSBOOKING_MANIFEST': str(MANIFEST_PATH), 'BUSBOOKING_TOKENS': str(TOKENS_PATH),
              'BUSBOOKING_HOT_TRIP_COUNT': str(HOT_TRIP_COUNT)})

hot_ids = ','.join(str(int(x)) for x in DATA['hot_trip_ids'])
SEAT_IMBALANCE = int(db_value(f"""
SELECT count(*) FROM (
    SELECT t.id
    FROM trips t
    JOIN vehicles v ON v.id = t.vehicle_id
    LEFT JOIN bookings b ON b.trip_id = t.id AND b.status = 'CONFIRMED'
    WHERE t.id = ANY(ARRAY[{hot_ids}]::bigint[])
    GROUP BY t.id, t.available_seats, v.seat_capacity
    HAVING t.available_seats + COALESCE(sum(b.quantity), 0) <> v.seat_capacity
) inconsistent"""))
HOT_BOOKINGS = int(db_value(f"SELECT count(*) FROM bookings WHERE trip_id = ANY(ARRAY[{hot_ids}]::bigint[])"))
print('Bookings on hot trips:', HOT_BOOKINGS)
print('Trips with incorrect seat balance:', SEAT_IMBALANCE)
assert SEAT_IMBALANCE == 0, 'Phát hiện sai lệch số ghế trên chuyến hot'

## 12. Kịch bản đăng ký: SMTP hoạt động và SMTP treo

- `register_smtp_ok`: Mailpit nhận mail bình thường.
- `register_smtp_hang`: thay Mailpit bằng một server giả nhận kết nối nhưng không trả lời, giống mail server bị treo.

Backend chạy với `AUTH_RATE_REGISTER_LIMIT` đã nâng (xem cell cấu hình); giá trị này được ghi vào manifest.

In [ ]:
import urllib.request, tarfile, io

MAILPIT_VERSION = 'v1.32.0'
mailpit_bin = WORK / 'mailpit'
mailpit_proc = None
blackhole_proc = None

def port_open(port_no):
    try:
        with socket.create_connection(('127.0.0.1', port_no), timeout=2):
            return True
    except OSError:
        return False

def wait_port(port_no, want_open, seconds=15):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        if port_open(port_no) == want_open:
            return
        time.sleep(0.5)
    raise RuntimeError(f'Cổng {port_no} không {"mở" if want_open else "đóng"} sau {seconds}s')

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()

if not mailpit_bin.exists():
    url = f'https://github.com/axllent/mailpit/releases/download/{MAILPIT_VERSION}/mailpit-linux-amd64.tar.gz'
    archive_data = urllib.request.urlopen(url, timeout=90).read()
    with tarfile.open(fileobj=io.BytesIO(archive_data), mode='r:gz') as tar:
        member = next(m for m in tar.getmembers() if m.isfile() and Path(m.name).name == 'mailpit')
        with tar.extractfile(member) as source, open(mailpit_bin, 'wb') as target:
            shutil.copyfileobj(source, target)
    mailpit_bin.chmod(0o755)

# SMTP hoạt động
mailpit_proc = subprocess.Popen([str(mailpit_bin), '--listen', '127.0.0.1:8025', '--smtp', '127.0.0.1:1025'],
                                cwd=str(WORK), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
wait_port(1025, True)
run_scenario('register_smtp_ok', LT / 'scripts/register_locust.py', LEVELS['register'], {})
MAILS_RECEIVED = requests.get('http://127.0.0.1:8025/api/v1/messages', params={'limit': 1}, timeout=10).json().get('total')
print('Mail Mailpit đang giữ (Mailpit tự xóa bớt mail cũ, không phải tổng số đã gửi):', MAILS_RECEIVED)

# SMTP treo
stop(mailpit_proc)
wait_port(1025, False)
blackhole_proc = subprocess.Popen([sys.executable, str(LT / 'scripts/smtp_blackhole.py'), '1025'],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
wait_port(1025, True)
try:
    run_scenario('register_smtp_hang', LT / 'scripts/register_locust.py', LEVELS['register'], {})
finally:
    stop(blackhole_proc)

PENDING_ACCOUNTS = int(db_value("SELECT count(*) FROM users WHERE email LIKE 'lt-reg-%@loadtest.example' AND NOT email_verified"))
print('Tài khoản đăng ký còn ở trạng thái chờ xác minh:', PENDING_ACCOUNTS)

## 13. Tổng hợp

Mỗi dòng là trung vị của `RUNS` lượt. `request = ALL` là tổng của cả kịch bản; các dòng khác tách theo loại request. `backend_died_runs` là số lượt mà backend không còn phản hồi khi kết thúc.

In [ ]:
runs_df = pd.DataFrame(RAW)
runs_df.to_csv(RAW_DIR / 'phase2_runs.csv', index=False)
median_cols = ['requests', 'failures', 'error_pct', 'rps', 'avg_ms', 'p50_ms', 'p95_ms', 'p99_ms',
               'java_cpu_avg_pct', 'java_rss_max_mb', 'pool_active_max', 'pool_pending_max', 'lock_wait_avg_ms']
group = runs_df.groupby(['scenario', 'users', 'request'], sort=False)
summary = group[median_cols].median().round(2)
summary['runs'] = group['run'].count()
summary['backend_died_runs'] = group['backend_died'].sum().astype(int)
summary = summary.reset_index()
summary.insert(0, 'stage', STAGE)
summary.to_csv(OUTPUT / 'phase2_summary.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 100)
display(summary[['scenario', 'users', 'request', 'rps', 'p50_ms', 'p95_ms', 'p99_ms', 'error_pct',
                 'java_cpu_avg_pct', 'java_rss_max_mb', 'pool_pending_max', 'lock_wait_avg_ms', 'backend_died_runs']])

## 14. Manifest môi trường và đóng gói

Tải file ZIP từ mục **Output** của Kaggle và lưu vào `load-tests/results/` trong repository. Trong ZIP:\n\n- `REPORT.md`: **file duy nhất cần đọc**, gồm bảng số liệu để đưa vào báo cáo.\n- `phase2_summary.csv`: cùng số liệu ở dạng bảng tính, để vẽ biểu đồ hoặc so sánh trước và sau.\n- `manifest.json`: cấu hình máy và phiên bản đã đo (không chứa mật khẩu hay token).\n- `raw/`: log và CSV chi tiết của từng lượt; chỉ mở khi cần tra cứu một lượt cụ thể.

In [ ]:
import datetime
try:
    cpu_model = next((line.split(':', 1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')), 'unknown')
except Exception:
    cpu_model = 'unknown'
log_text = backend_log.read_text(errors='replace')
manifest = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'stage': STAGE,
    'jar_filename': JAR_PATH.name, 'jar_sha256': JAR_SHA256,
    'scripts_source': SCRIPTS_SOURCE, 'scripts_commit': SCRIPTS_COMMIT, 'script_sha256_prefix': SCRIPT_HASHES,
    'java_version': subprocess.run([str(JAVA_BIN), '-version'], capture_output=True, text=True).stderr.splitlines()[0],
    'jvm_options': JVM_OPTIONS,
    'postgres_client_version': POSTGRES_VERSION,
    'locust_version': subprocess.check_output([sys.executable, '-m', 'locust', '--version'], text=True).strip(),
    'psutil_version': psutil.__version__,
    'mailpit_version': MAILPIT_VERSION,
    'platform': platform.platform(), 'cpu_model': cpu_model, 'logical_cpus': os.cpu_count(),
    'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 2),
    'run_seconds': RUN_SECONDS, 'runs_per_level': RUNS, 'warmup_seconds': WARMUP_SECONDS,
    'levels': LEVELS, 'hot_trip_count': HOT_TRIP_COUNT,
    'dataset': DATASET_COUNTS,
    'auth_rate_register_limit': AUTH_RATE_REGISTER_LIMIT,
    'actuator_metrics_available': HAS_METRICS,
    'backend_starts': backend_starts,
    'backend_out_of_memory_errors': log_text.count('java.lang.OutOfMemoryError'),
    'hot_trip_bookings': HOT_BOOKINGS, 'hot_trip_seat_imbalance': SEAT_IMBALANCE,
    'register_mails_received': MAILS_RECEIVED, 'register_pending_accounts': PENDING_ACCOUNTS,
    'notes': 'Backend, PostgreSQL và Locust chạy chung một máy. CPU Java là Process.cpu_percent(), chưa chia cho số CPU logic.',
}
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(manifest, indent=2, ensure_ascii=False))

# REPORT.md: file duy nhất cần đọc khi viết báo cáo
def fmt(value, digits=1):
    return '-' if pd.isna(value) else f'{value:,.{digits}f}'.replace(',', ' ').replace('.', ',')

lines = [
    f'# Kết quả load test Pha 2: `{STAGE}`',
    '',
    f"- Thời điểm: {manifest['timestamp_utc'][:19]} UTC",
    f"- JAR: `{JAR_PATH.name}` (SHA256 `{JAR_SHA256[:16]}…`)",
    f"- Máy: {cpu_model}, {os.cpu_count()} CPU logic, {manifest['ram_total_gb']} GB RAM",
    f"- Dữ liệu: {fmt(DATASET_COUNTS['trips'], 0)} chuyến xe, {fmt(DATASET_COUNTS['bookings'], 0)} booking",
    f'- Mỗi mức tải: {RUNS} lượt x {RUN_SECONDS} giây, lấy trung vị',
    '',
    '## Bảng số liệu',
    '',
    '| Kịch bản | Người dùng ảo | Request | RPS | p50 (ms) | p95 (ms) | p99 (ms) | Lỗi (%) |',
    '|---|---|---|---|---|---|---|---|',
]
for _, r in summary.iterrows():
    request = '**Tổng**' if r['request'] == 'ALL' else r['request']
    lines.append(f"| {r['scenario']} | {r['users']} | {request} | {fmt(r['rps'])} | {fmt(r['p50_ms'], 0)} | {fmt(r['p95_ms'], 0)} | {fmt(r['p99_ms'], 0)} | {fmt(r['error_pct'], 2)} |")
lines += [
    '',
    '## Tài nguyên và tranh chấp (dòng tổng của mỗi kịch bản)',
    '',
    '| Kịch bản | Người dùng ảo | CPU Java (%) | RAM Java tối đa (MB) | Request chờ kết nối DB (tối đa) | Chờ khóa trung bình (ms) | Số lượt backend chết |',
    '|---|---|---|---|---|---|---|',
]
for _, r in summary[summary['request'] == 'ALL'].iterrows():
    lines.append(f"| {r['scenario']} | {r['users']} | {fmt(r['java_cpu_avg_pct'])} | {fmt(r['java_rss_max_mb'], 0)} | {fmt(r['pool_pending_max'], 0)} | {fmt(r['lock_wait_avg_ms'], 2)} | {r['backend_died_runs']} |")
lines += [
    '',
    '## Kiểm tra tính đúng',
    '',
    f'- Chuyến "hot": {HOT_BOOKINGS} booking, {SEAT_IMBALANCE} chuyến lệch số ghế.',
    f'- Đăng ký: {PENDING_ACCOUNTS} tài khoản được tạo và còn chờ xác minh (gồm cả các lượt đăng ký bị lỗi khi SMTP treo).',
    f"- Backend khởi động {backend_starts} lần; {manifest['backend_out_of_memory_errors']} lỗi hết bộ nhớ trong log.",
    '',
    'CPU Java là tổng trên mọi nhân (100% = một nhân). Backend, PostgreSQL và Locust chạy chung một máy.',
    f'Giới hạn đăng ký được nâng lên {AUTH_RATE_REGISTER_LIMIT} request/phút để đo kịch bản đăng ký.',
]
(OUTPUT / 'REPORT.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))

stop_backend()
archive = shutil.make_archive(str(WORK / OUTPUT.name), 'zip', root_dir=str(OUTPUT))
print('Report:', archive)